<a href="https://colab.research.google.com/github/sutida254526/parttime-scheduler/blob/main/app_M2T1P8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install pulp

In [36]:
all_employees_avail = {
    1: {  1: [1,2,3,4],
          2: [],
          3: [1,2,3,4],
          4: [],
          5: [],
          6: []
       },
    2: {  1: [1,2,3,4],
          2: [1,2,3,4],
          3: [],
          4: [],
          5: [1,2,3,4],
          6: [1,2,3,4]
       },
    3: {  1: [2,3,4],
          2: [2,3,4],
          3: [],
          4: [2,3,4],
          5: [],
          6: [2,3,4]
       },
    4: {  1: [1,2,3,4],
          2: [1,2,3,4],
          3: [],
          4: [],
          5: [],
          6: [1,2,3,4]
       },
    5: {  1: [1,2,3,4],
          2: [1,2,3,4],
          3: [],
          4: [],
          5: [],
          6: [1,2,3,4]
       },
    6: {  1: [2,3,4],
          2: [],
          3: [2,3,4],
          4: [],
          5: [2,3,4],
          6: []
       },
    7: {  1: [2,3,4],
          2: [],
          3: [2,3,4],
          4: [2,3,4],
          5: [2,3,4],
          6: [2,3,4]
       },
    8: {  1: [2,3,4],
          2: [2,3,4],
          3: [2,3,4],
          4: [2,3,4],
          5: [],
          6: []
       },
    9: {  1: [3,4],
          2: [],
          3: [3,4],
          4: [3,4],
          5: [3,4],
          6: [3,4]
       },
    10: { 1: [1,2,3,4],
          2: [1,2,3,4],
          3: [],
          4: [],
          5: [1,2,3,4],
          6: []
       },
    11: { 1: [1,2,3,4],
          2: [],
          3: [],
          4: [1,2,3,4],
          5: [1,2,3,4],
          6: [1,2,3,4]
       },
    12: { 1: [1,2,3,4],
          2: [1,2,3,4],
          3: [],
          4: [],
          5: [],
          6: [1,2,3,4]
       },
    13: { 1: [2,3,4],
          2: [],
          3: [2,3,4],
          4: [],
          5: [2,3,4],
          6: [2,3,4]
       },
    14: { 1: [1,2,3,4],
          2: [1,2,3,4],
          3: [1,2,3,4],
          4: [],
          5: [1,2,3,4],
          6: []
       },
    15: { 1: [1,2,3,4],
          2: [],
          3: [],
          4: [1,2,3,4],
          5: [],
          6: []
       },
    16: { 1: [2,3,4],
          2: [2,3,4],
          3: [],
          4: [],
          5: [2,3,4],
          6: [2,3,4]
       },
}


data = {
    "num_employees": 16,
    "W_per_t": {1: 1,
                2: 2,
                3: 3,
                4: 2},
    "max_shift_i": 3,
    "P_idt": all_employees_avail,
    "cost_per_shift": {1: 320, 2: 160, 3: 160, 4: 160}
    }

In [34]:
import pulp
import random, numpy as np


def solver_parttime(data):
    random.seed(42)
    np.random.seed(42)


    num_employees = data["num_employees"]
    W_per_t = data["W_per_t"]
    max_shift_i = data["max_shift_i"]
    all_employees_avail = data["P_idt"]
    cost_per_shift = data["cost_per_shift"]

    # ====== Sets ======
    I = range(1, num_employees+1)   # พนักงานพาร์ทไทม์ i = 1..16
    D = range(1, 7)    # วัน d = 1..6 (1=อาทิตย์,2=จันทร์,...,6=ศุกร์)
    T = range(1, 5)    # ช่วงเวลา t = 1..4

    # ====== Parameters ======
    C_it = {(i,t): cost_per_shift[t] for i in I for t in T}
    W_dt = {(d,t): W_per_t[t] for d in D for t in T}
    maxShift_i = {i: max_shift_i for i in I}



    #  สร้างตัวแปรตัดสินใจ (Decision Variables)
    # -----------------------------
    M = pulp.LpVariable.dicts("main", (I, D, T), cat="Binary")     # พนักงานหลัก
    B = pulp.LpVariable.dicts("backup", (I, D, T), cat="Binary")   # พนักงานสำรอง
    A = pulp.LpVariable.dicts("A", (I, D, T), cat="Binary")   # การมอบหมายงานจริง


    model = pulp.LpProblem("PartTime_Scheduling", pulp.LpMinimize)

      #  ฟังก์ชันวัตถุประสงค์ (Objective Function)
    model += pulp.lpSum(C_it[(i, t)] * M[i][d][t] for i in I for d in D for t in T)



    #P_idt = {(i,d,t): 0 for i in I for d in D for t in T}
    P_idt = {}
    for i in I:
        for d in D:
            for t in T:
                if t in all_employees_avail[i][d]:
                    P_idt[(i,d,t)] = 1
                else:
                    P_idt[(i,d,t)] = 0

    ##  เงื่อนไข (Constraints)
    # -----------------------------


    # 5.1 จำนวนพนักงานหลักตามช่วงเวลา
    for d in D:
        for t in T:
            model += pulp.lpSum(M[i][d][t] for i in I) >= W_dt[(d, t)]

    # 5.2 จำนวนพนักงานสำรองตามช่วงเวลา
    for d in D:
        for t in T:
            model += pulp.lpSum(B[i][d][t] for i in I) >= W_dt[(d, t)]

    # 5.3 แต่ละคนต้องทำงานหลักอย่างน้อย 1 กะ/สัปดาห์
    for i in I:
        model += pulp.lpSum(M[i][d][t] for d in D for t in T) >= 1

    # 5.4 แต่ละคนต้องทำงานสำรองอย่างน้อย 1 กะ/สัปดาห์
    for i in I:
        model += pulp.lpSum(B[i][d][t] for d in D for t in T) >= 1

    # 5.5 จำกัดกะสูงสุดต่อสัปดาห์ (หลัก)
    for i in I:
        model += pulp.lpSum(M[i][d][t] for d in D for t in T) <= maxShift_i[i]

    # 5.6 จำกัดกะสูงสุดต่อสัปดาห์ (สำรอง)
    for i in I:
        model += pulp.lpSum(B[i][d][t] for d in D for t in T) <= maxShift_i[i]

    # 5.7 ห้ามเป็นหลักและสำรองในช่วงเวลาเดียวกัน
    for i in I:
        for d in D:
            for t in T:
                model += M[i][d][t] + B[i][d][t] <= 1

    # 5.8 ต้องเลือกจากพนักงานที่ว่าง
    for i in I:
        for d in D:
            for t in T:
              model += M[i][d][t] <= P_idt[(i, d, t)]
              model += B[i][d][t] <= P_idt[(i, d, t)]


    # 5.9 ความต่างของงานที่แต่ละคนได้รับไม่เกิน 1 กะ
    for i in I:
        for k in I:
            if i != k:
                model += (
                    pulp.lpSum(M[i][d][t] + B[i][d][t] for d in D for t in T) -
                    pulp.lpSum(M[k][d][t] + B[k][d][t] for d in D for t in T)
                ) <= 1
                model += (
                    pulp.lpSum(M[k][d][t] + B[k][d][t] for d in D for t in T) -
                    pulp.lpSum(M[i][d][t] + B[i][d][t] for d in D for t in T)
                ) <= 1

    # ====== Solve ======
    solver = pulp.PULP_CBC_CMD(msg=False, options=['randomSeed=42'])
    model.solve(solver)

    main_schedule = {(d,t): [i for i in I if pulp.value(M[i][d][t])==1] for d in D for t in T}
    backup_schedule = {(d,t): [i for i in I if pulp.value(B[i][d][t])==1] for d in D for t in T}

    main_count = {i: sum(pulp.value(M[i][d][t]) for d in D for t in T) for i in I}

    total_cost = pulp.value(model.objective)

    print(f"\nค่าใช้จ่าย: {total_cost} บาท/สัปดาห์\n")
    print("=== Main Schedule ===")
    for d in D:
        print(f"Day {d}:\n", end=" ")
        for t in T:
            emp_list = main_schedule[(d,t)]
            print(f"Shift {t}: {emp_list}\n", end="   ")
        print()

    print("\n=== Backup Schedule ===")
    for d in D:
        print(f"Day {d}:\n", end=" ")
        for t in T:
            emp_list = backup_schedule[(d,t)]
            print(f"Shift {t}: {emp_list}\n", end="   ")
        print()

    return {"main": main_schedule,
            "backup": backup_schedule,
            "main_count": main_count,
            "total_cost": total_cost,
            "status": pulp.LpStatus[model.status]}


In [35]:
result = solver_parttime(data)


ค่าใช้จ่าย: 8640.0 บาท/สัปดาห์

=== Main Schedule ===
Day 1:
 Shift 1: [1]
   Shift 2: [8, 10]
   Shift 3: [7, 14, 16]
   Shift 4: [5, 8]
   
Day 2:
 Shift 1: [10]
   Shift 2: [2, 14]
   Shift 3: [3, 4, 16]
   Shift 4: [8, 12]
   
Day 3:
 Shift 1: [1]
   Shift 2: [1, 6]
   Shift 3: [9, 13, 14]
   Shift 4: [6, 13]
   
Day 4:
 Shift 1: [15]
   Shift 2: [3, 15]
   Shift 3: [9, 11, 15]
   Shift 4: [3, 11]
   
Day 5:
 Shift 1: [2]
   Shift 2: [7, 16]
   Shift 3: [7, 10, 13]
   Shift 4: [6, 9]
   
Day 6:
 Shift 1: [4]
   Shift 2: [11, 12]
   Shift 3: [4, 5, 12]
   Shift 4: [2, 5]
   

=== Backup Schedule ===
Day 1:
 Shift 1: [5]
   Shift 2: [1, 15]
   Shift 3: [1, 2, 15]
   Shift 4: [6, 12]
   
Day 2:
 Shift 1: [5]
   Shift 2: [3, 4]
   Shift 3: [2, 10, 12]
   Shift 4: [3, 16]
   
Day 3:
 Shift 1: [14]
   Shift 2: [13, 14]
   Shift 3: [1, 6, 8]
   Shift 4: [7, 9]
   
Day 4:
 Shift 1: [11]
   Shift 2: [8, 11]
   Shift 3: [3, 7, 8]
   Shift 4: [7, 15]
   
Day 5:
 Shift 1: [10]
   Shift 2: [6,

In [ ]:
print("==== พนักงานพาร์ทไทม์: ตัวจริง ====")
for i in I:
    for d in D:
        for t in T:
            # เช็คว่าพนักงาน i ทำงานเป็นตัวจริงในวัน d ช่วงเวลา t หรือไม่
            if pulp.value(M[i][d][t]) == 1:
                print(f"พนักงานพาร์ทไทม์ {i} ทำงานเป็น 'ตัวจริง' วัน {d} ช่วงเวลา {t}")

print("\n==== พนักงานพาร์ทไทม์: ตัวสำรอง ====")
for i in I:
    for d in D:
        for t in T:
            # เช็คว่าพนักงาน i ทำงานเป็นตัวสำรองในวัน d ช่วงเวลา t หรือไม่
            if pulp.value(B[i][d][t]) == 1:
                print(f"พนักงานพาร์ทไทม์ {i} เป็น 'ตัวสำรอง' วัน {d} ช่วงเวลา {t}")


In [ ]:
# ====== แสดงผลลัพธ์ ======
days = {1: "อาทิตย์", 2: "จันทร์", 3: "อังคาร", 4: "พุธ", 5: "พฤหัส", 6: "ศุกร์"}
shifts = {1: "กะ 1", 2: "กะ 2", 3: "กะ 3", 4: "กะ 4"}

for d in D:
    print(f"\n=== วัน{days[d]} ===")
    for t in T:
        main_workers = [i for i in I if pulp.value(M[i][d][t]) == 1]
        backup_workers = [i for i in I if pulp.value(B[i][d][t]) == 1]
        print(f" {shifts[t]}:")
        print(f"   หลัก   → {main_workers if main_workers else 'ไม่มี'}")
        print(f"   สำรอง → {backup_workers if backup_workers else 'ไม่มี'}")


In [ ]:
print("=== ผลลัพธ์พนักงานตัวจริง (Main) ===\n")

total_all = 0  # เก็บค่าตอบแทนรวมของทุกคน

for i in I:
    shifts_per_t = {t: 0 for t in T}
    total_shifts = 0

    for d in D:
        for t in T:
            if pulp.value(M[i][d][t]) > 0.5:
                shifts_per_t[t] += 1
                total_shifts += 1

    total_pay = sum(C_it[(i, t)] * shifts_per_t[t] for t in T)
    total_all += total_pay  # บวกค่าตอบแทนแต่ละคนเข้า total รวม

    print(f"พนักงาน {i}:")
    for t in T:
        print(f"  ▪️ กะ {t}: {shifts_per_t[t]} วัน")
    print(f"  ➤ รวมทั้งหมด {total_shifts} กะ  |  ค่าตอบแทนรวม {total_pay:,.0f} บาท\n")

# แสดงค่าตอบแทนรวมของทุกคน
print("="*50)
print(f"💰 ค่าตอบแทนรวมของพนักงานตัวจริงทั้งหมด 16 คน = {total_all:,.0f} บาท")
print("="*50)


In [ ]:
# ====== สรุปผลรวมต่อกะ ======
print("\n=== สรุปจำนวนครั้งที่พนักงานทำงานในแต่ละกะ (ตัวจริง) ===")
for t in T:
    total_main = sum(pulp.value(M[i][d][t]) for i in I for d in D)
    print(f"กะที่ {t}: {int(total_main)} กะ")

print("\n=== สรุปจำนวนครั้งที่พนักงานทำงานในแต่ละกะ (สำรอง) ===")
for t in T:
    total_backup = sum(pulp.value(B[i][d][t]) for i in I for d in D)
    print(f"กะที่ {t}: {int(total_backup)} กะ")


# ====== สรุปค่าใช้จ่าย ======
total_main_cost = sum(C_it[(i, t)] * pulp.value(M[i][d][t]) for i in I for d in D for t in T)
total_backup_cost = sum(C_it[(i, t)] * pulp.value(B[i][d][t]) for i in I for d in D for t in T)

print("\n=== สรุปค่าใช้จ่าย ===")
print(f"ค่าใช้จ่ายพนักงานตัวจริงรวม: {int(total_main_cost)} บาท")
print(f"ค่าใช้จ่ายพนักงานสำรองรวม: {int(total_backup_cost)} บาท")
print(f"ค่าใช้จ่ายรวมทั้งหมด: {int(total_main_cost + total_backup_cost)} บาท")

# ====== สรุปค่าใช้จ่ายรายกะ (ตัวจริง) ======
print("\n=== ค่าใช้จ่ายรายกะ (ตัวจริง) ===")
for t in T:
    cost_t = sum(C_it[(i, t)] * pulp.value(M[i][d][t]) for i in I for d in D)
    print(f"กะที่ {t}: {int(cost_t)} บาท")

# ====== สรุปค่าใช้จ่ายรายกะ (สำรอง) ======
print("\n=== ค่าใช้จ่ายรายกะ (สำรอง) ===")
for t in T:
    cost_t = sum(C_it[(i, t)] * pulp.value(B[i][d][t]) for i in I for d in D)
    print(f"กะที่ {t}: {int(cost_t)} บาท")


In [ ]:
# ====== แสดงผลลัพธ์แบบอ่านง่าย ======

day_names = {
    1: "อาทิตย์", 2: "จันทร์", 3: "อังคาร",
    4: "พุธ", 5: "พฤหัสบดี", 6: "ศุกร์"
}

print("\n=== พนักงานตัวจริงแต่ละวัน ===")
for d in D:
    print(f"\nพนักงานตัวจริง วันที่ {d} ({day_names[d]})")
    for t in T:
        mains = [i for i in I if pulp.value(M[i][d][t]) == 1]
        print(f"  กะ {t}: {mains if mains else 'ไม่มี'}")

print("\n=== พนักงานสำรองแต่ละวัน ===")
for d in D:
    print(f"\nพนักงานสำรอง วันที่ {d} ({day_names[d]})")
    for t in T:
        backups = [i for i in I if pulp.value(B[i][d][t]) == 1]
        print(f"  กะ {t}: {backups if backups else 'ไม่มี'}")
